# SINDy + DAgger — Closed-Loop Dataset Aggregation

**Goal:** Use DAgger (Dataset Aggregation, Ross et al. 2011) to push SINDy R² close to
1 by targeting the *closed-loop* failure distribution — the states the learned policy
actually visits — rather than just the expert's near-equilibrium trajectory.

## Why feature engineering alone is not enough

Trig terms, post-hoc refit, and SR3 all improve the polynomial's fit on the *expert's*
state distribution. But the expert visits near-equilibrium states; the learned policy,
once deployed closed-loop, visits different states — especially under perturbation.
No amount of offline fitting on expert data can fix an error that only manifests during
the learned policy's own rollouts.

## DAgger algorithm

1. Fit SINDy on the expert dataset $(X_0, U_0)$ → initial policy $\pi_0$
2. Roll out $\pi_i$ in the environment; collect visited states $X_i^\text{roll}$
3. Query expert at those states: $U_i^\text{roll} = \pi^\star(X_i^\text{roll})$
4. Aggregate: $X_{i+1} = X_i \cup X_i^\text{roll}$, $U_{i+1} = U_i \cup U_i^\text{roll}$
5. Refit SINDy on the aggregated dataset → $\pi_{i+1}$
6. Repeat

Each iteration adds states the *current* policy actually encounters, directly
covering the closed-loop failure modes.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pysindy as ps
import gymnasium as gym
import torch
from sklearn.preprocessing import StandardScaler
from joblib import Parallel, delayed
from tqdm.auto import tqdm
from collections import defaultdict
from stable_baselines3 import PPO

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
from plotting_utils import plot_eval_bars, render_episode, plot_pca_coverage

DATA_PATH  = pathlib.Path("../data/baseline/trajectories_baseline.npz")
CHECKPOINT = "../data/baseline/checkpoints/best_model.zip"
ENV_ID     = "InvertedDoublePendulum-v5"

# ── DAgger config ──────────────────────────────────────────────────────────────
DAGGER_ITERS       = 8      # number of aggregation rounds
DAGGER_EPISODES    = 5      # rollout episodes per round
SINDY_DEGREE       = 2      # polynomial degree
SINDY_THRESHOLD    = 0.01
USE_TRIG           = True   # append sin/cos of angles to state-6
USE_SCALER         = True   # StandardScaler on inputs
N_EVAL             = 20
NOISE_LEVELS       = [0.01, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5]

_env       = gym.make(ENV_ID)
MAX_STEPS  = _env.spec.max_episode_steps
DT         = _env.unwrapped.dt
EVAL_NOISE = getattr(_env.unwrapped, "reset_noise_scale",
                     getattr(_env.unwrapped, "_reset_noise_scale", 0.1))
_env.close()
print(f"MAX_STEPS={MAX_STEPS}  DT={DT}  EVAL_NOISE={EVAL_NOISE}")

MAX_STEPS=1000  DT=0.05  EVAL_NOISE=0.1


In [2]:
# ── Data helpers ───────────────────────────────────────────────────────────────
def obs_to_state6(obs):
    return np.array([
        obs[0],
        np.arctan2(obs[1], obs[3]), np.arctan2(obs[2], obs[4]),
        obs[5], obs[6], obs[7],
    ], dtype=np.float64)

def state6_to_obs9(s):
    x, th1, th2, xd, th1d, th2d = s
    return np.array([
        x, np.sin(th1), np.sin(th2), np.cos(th1), np.cos(th2),
        xd, th1d, th2d, 0.0,
    ], dtype=np.float32)

def to_features(X6):
    """state-6 → feature input (appends trig if USE_TRIG)."""
    if USE_TRIG:
        return np.hstack([
            X6,
            np.sin(X6[:, 1:2]), np.cos(X6[:, 1:2]),
            np.sin(X6[:, 2:3]), np.cos(X6[:, 2:3]),
        ])
    return X6


# ── Load expert data ───────────────────────────────────────────────────────────
data  = np.load(DATA_PATH)
X6_0  = data["X"].astype(np.float64)
U_0   = data["U"].astype(np.float64)
if U_0.ndim == 1:
    U_0 = U_0.reshape(-1, 1)

print(f"Expert dataset: {len(X6_0):,} transitions")

# ── Expert model (for labelling DAgger states) ─────────────────────────────────
expert = PPO.load(CHECKPOINT)
device = expert.policy.device

def query_expert(X6_states):
    """Query PPO expert at a batch of state-6 vectors; returns (N,1) actions."""
    obs9 = np.array([state6_to_obs9(s) for s in X6_states], dtype=np.float32)
    obs_t = torch.as_tensor(obs9).to(device)
    with torch.no_grad():
        u = (expert.policy.get_distribution(obs_t)
             .get_actions(deterministic=True)
             .cpu().numpy().reshape(-1, 1).astype(np.float64))
    return u

Expert dataset: 50,000 transitions


In [3]:
# ── Fit / eval helpers ─────────────────────────────────────────────────────────
def fit_sindy(X6, U, scaler=None):
    """Fit STLSQ on feature-engineered X6; return fit dict."""
    Xf = to_features(X6)
    if scaler is None and USE_SCALER:
        scaler = StandardScaler()
    Xs = scaler.fit_transform(Xf) if scaler is not None else Xf

    lib   = ps.PolynomialLibrary(degree=SINDY_DEGREE)
    lib.fit(Xs)
    Theta = lib.transform(Xs)
    coeff = ps.STLSQ(threshold=SINDY_THRESHOLD).fit(Theta, U).coef_

    # post-hoc LS refit on surviving terms
    active = np.where(np.abs(coeff).flatten() > 1e-8)[0]
    if len(active) > 0:
        c_ls, _, _, _ = np.linalg.lstsq(Theta[:, active], U, rcond=None)
        coeff[:, active] = c_ls.T

    nf   = Theta.shape[1]
    nz   = int(np.sum(np.abs(coeff) > 1e-8))
    up   = np.asarray((Theta @ coeff.T).flatten())
    rmse = float(np.sqrt(np.mean((U.flatten() - up) ** 2)))
    r2   = float(1.0 - np.sum((U.flatten() - up)**2) /
                        np.sum((U.flatten() - U.mean())**2))
    return dict(lib=lib, coeff=coeff, scaler=scaler,
                nf=nf, nz=nz, rmse=rmse, r2=r2, n_train=len(X6))


def make_policy(fit):
    lib, coeff, scaler = fit["lib"], fit["coeff"], fit["scaler"]
    def policy(obs):
        s6 = obs_to_state6(obs)
        xf = to_features(s6.reshape(1, -1))
        xs = scaler.transform(xf) if scaler is not None else xf
        u  = float(np.asarray(fit["lib"].transform(xs) @ coeff.T).flatten()[0])
        return np.array([np.clip(u, -1.0, 1.0)], dtype=np.float32)
    return policy


def rollout_states(policy_fn, n_episodes, noise, seed=0):
    """Roll out policy; return (X6_visited, U_expert_labels)."""
    all_states = []
    for ep in range(n_episodes):
        env = gym.make(ENV_ID, reset_noise_scale=noise)
        obs, _ = env.reset(seed=seed + ep)
        done = False
        while not done:
            all_states.append(obs_to_state6(obs))
            obs, _, terminated, truncated, _ = env.step(policy_fn(obs))
            done = terminated or truncated
        env.close()
    X6_roll = np.array(all_states)
    U_roll  = query_expert(X6_roll)
    return X6_roll, U_roll


def eval_policy(policy_fn, noise, n_episodes=N_EVAL, seed=100):
    lens = []
    for ep in range(n_episodes):
        env = gym.make(ENV_ID, reset_noise_scale=noise)
        obs, _ = env.reset(seed=seed + ep)
        ep_l = 0; done = False
        while not done:
            obs, _, terminated, truncated, _ = env.step(policy_fn(obs))
            ep_l += 1; done = terminated or truncated
        lens.append(ep_l); env.close()
    return np.array(lens)


def r2_on_expert(fit):
    """Compute R² of fit on original expert dataset (honest metric)."""
    Xf = to_features(X6_0)
    Xs = fit["scaler"].transform(Xf) if fit["scaler"] is not None else Xf
    Theta = fit["lib"].transform(Xs)
    up = np.asarray((Theta @ fit["coeff"].T).flatten())
    return float(1.0 - np.sum((U_0.flatten() - up)**2) /
                        np.sum((U_0.flatten() - U_0.mean())**2))

---

## DAgger loop

In [4]:
# ── Iteration 0: fit on expert data only ──────────────────────────────────────
X6_agg, U_agg = X6_0.copy(), U_0.copy()

history = []   # track metrics per iteration

fit_i    = fit_sindy(X6_agg, U_agg)
policy_i = make_policy(fit_i)
lens_i   = eval_policy(policy_i, EVAL_NOISE)

history.append({
    "iteration":  0,
    "n_train":    len(X6_agg),
    "R² (train)": round(fit_i["r2"], 4),
    "R² (expert)": round(r2_on_expert(fit_i), 4),
    "nonzero":    fit_i["nz"],
    "mean_steps": round(lens_i.mean(), 1),
    "pct_complete": round(100 * np.mean(lens_i == MAX_STEPS), 1),
})
print(f"Iter 0  n={len(X6_agg):>7,}  R²={fit_i['r2']:.4f}  "
      f"steps={lens_i.mean():.0f}/{MAX_STEPS}  "
      f"({100*np.mean(lens_i==MAX_STEPS):.0f}% complete)")

# ── DAgger iterations ─────────────────────────────────────────────────────────
for it in tqdm(range(1, DAGGER_ITERS + 1), desc="DAgger"):
    X6_roll, U_roll = rollout_states(
        policy_i, DAGGER_EPISODES, EVAL_NOISE, seed=it * 1000
    )
    X6_agg = np.vstack([X6_agg, X6_roll])
    U_agg  = np.vstack([U_agg,  U_roll])

    fit_i    = fit_sindy(X6_agg, U_agg)
    policy_i = make_policy(fit_i)
    lens_i   = eval_policy(policy_i, EVAL_NOISE)

    history.append({
        "iteration":   it,
        "n_train":     len(X6_agg),
        "R² (train)":  round(fit_i["r2"], 4),
        "R² (expert)": round(r2_on_expert(fit_i), 4),
        "nonzero":     fit_i["nz"],
        "mean_steps":  round(lens_i.mean(), 1),
        "pct_complete": round(100 * np.mean(lens_i == MAX_STEPS), 1),
    })
    print(f"Iter {it}  n={len(X6_agg):>7,}  R²={fit_i['r2']:.4f}  "
          f"steps={lens_i.mean():.0f}/{MAX_STEPS}  "
          f"({100*np.mean(lens_i==MAX_STEPS):.0f}% complete)")

hist_df = pd.DataFrame(history).set_index("iteration")
hist_df

Iter 0  n= 50,000  R²=0.8671  steps=1000/1000  (100% complete)


DAgger:   0%|          | 0/8 [00:00<?, ?it/s]

Iter 1  n= 55,000  R²=0.8667  steps=1000/1000  (100% complete)
Iter 2  n= 60,000  R²=0.8667  steps=1000/1000  (100% complete)
Iter 3  n= 65,000  R²=0.8665  steps=1000/1000  (100% complete)
Iter 4  n= 70,000  R²=0.8667  steps=1000/1000  (100% complete)
Iter 5  n= 75,000  R²=0.8667  steps=1000/1000  (100% complete)
Iter 6  n= 80,000  R²=0.8667  steps=1000/1000  (100% complete)
Iter 7  n= 85,000  R²=0.8667  steps=1000/1000  (100% complete)


KeyboardInterrupt: 

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

iters = hist_df.index.tolist()

axes[0].plot(iters, hist_df["R² (train)"],  "o-", color="steelblue",  lw=1.8, ms=6, label="R² (train)")
axes[0].plot(iters, hist_df["R² (expert)"], "s--", color="darkorange", lw=1.5, ms=6, label="R² (expert)")
axes[0].set_xlabel("DAgger iteration")
axes[0].set_ylabel("R²")
axes[0].set_title("Fit quality")
axes[0].legend(fontsize=9)

axes[1].plot(iters, hist_df["mean_steps"], "o-", color="steelblue", lw=1.8, ms=6)
axes[1].axhline(MAX_STEPS, color="green", ls="--", lw=1.5, label=f"max ({MAX_STEPS})")
axes[1].set_xlabel("DAgger iteration")
axes[1].set_ylabel("Mean episode length")
axes[1].set_title("Closed-loop performance")
axes[1].legend(fontsize=9)

axes[2].plot(iters, hist_df["nonzero"], "o-", color="darkorange", lw=1.8, ms=6)
axes[2].set_xlabel("DAgger iteration")
axes[2].set_ylabel("Nonzero terms")
axes[2].set_title("Sparsity")

plt.suptitle("DAgger convergence", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
_r2 = fit_i["r2"]
print(f"Final DAgger R² = {_r2:.4f}")
assert _r2 >= 0.95, (
    f"R²={_r2:.4f} is below 0.95 — fit quality is insufficient to cleanly demonstrate "
    f"the BC compounding-error phenomenon. Increase DAGGER_ITERS, add trig features, "
    f"or adjust SINDY_DEGREE/SINDY_THRESHOLD before proceeding."
)

---

## Final policy — perturbation robustness and BC demonstration

In [ ]:
final_policy = policy_i
final_fit    = fit_i

print(f"Final model  R²={final_fit['r2']:.4f}  nz={final_fit['nz']}  "
      f"n_train={final_fit['n_train']:,}")

render_episode(final_policy, ENV_ID, MAX_STEPS, DT,
               title=f"SINDy+DAgger — iter {DAGGER_ITERS}", seed=0)

In [ ]:
# perturbation sweep
def _run_ep(noise, seed):
    import numpy as np, gymnasium as gym
    lib, coeff, scaler = final_fit["lib"], final_fit["coeff"], final_fit["scaler"]
    def policy(obs):
        s6 = np.array([obs[0], np.arctan2(obs[1],obs[3]), np.arctan2(obs[2],obs[4]),
                       obs[5], obs[6], obs[7]], dtype=np.float64)
        xf = np.hstack([s6, np.sin(s6[1]), np.cos(s6[1]),
                            np.sin(s6[2]), np.cos(s6[2])]).reshape(1,-1) if USE_TRIG else s6.reshape(1,-1)
        xs = scaler.transform(xf) if scaler is not None else xf
        u  = float(np.asarray(lib.transform(xs) @ coeff.T).flatten()[0])
        return np.array([np.clip(u,-1.,1.)], dtype=np.float32)
    env = gym.make(ENV_ID, reset_noise_scale=noise)
    obs, _ = env.reset(seed=seed)
    ep_l = 0; done = False
    while not done:
        obs, _, t, tr, _ = env.step(policy(obs)); ep_l+=1; done=t or tr
    env.close(); return ep_l

sweep_tasks = [(σ, ep) for σ in NOISE_LEVELS for ep in range(10)]
sweep_lens  = list(tqdm(
    Parallel(n_jobs=-1, prefer="threads", return_as="generator")(
        delayed(_run_ep)(σ, seed) for σ, seed in sweep_tasks
    ), total=len(sweep_tasks), desc="Sweep", unit="ep"
))

buckets = defaultdict(list)
for (σ, _), l in zip(sweep_tasks, sweep_lens):
    buckets[σ].append(l)

sweep_means = {σ: float(np.mean(buckets[σ])) for σ in NOISE_LEVELS}
for σ, m in sweep_means.items():
    print(f"  noise_std={σ:.2f}  mean_ep_len={m:.0f} / {MAX_STEPS}")

BC_DEMO_NOISE = next((σ for σ in NOISE_LEVELS if sweep_means[σ] < 800), NOISE_LEVELS[-1])
print(f"\nBC_DEMO_NOISE = {BC_DEMO_NOISE}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(NOISE_LEVELS, [sweep_means[σ] for σ in NOISE_LEVELS],
        "o-", color="steelblue", lw=1.8, ms=6)
ax.axhline(MAX_STEPS, color="green",  ls="--", lw=1.5, label=f"max ({MAX_STEPS}")
ax.axvline(BC_DEMO_NOISE, color="crimson", ls=":", lw=1.5, label=f"BC demo (σ={BC_DEMO_NOISE})")
ax.set_xlabel("Reset noise std"); ax.set_ylabel("Mean episode length")
ax.set_title("SINDy+DAgger — perturbation robustness")
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

In [ ]:
# failing rollout + PCA — does the trajectory now leave the training cloud?
env = gym.make(ENV_ID, reset_noise_scale=BC_DEMO_NOISE)
obs, _ = env.reset(seed=42)
states_demo = []
done = False
while not done:
    states_demo.append(obs_to_state6(obs))
    obs, _, terminated, truncated, _ = env.step(final_policy(obs))
    done = terminated or truncated
env.close()
states_demo = np.array(states_demo)
print(f"Demo rollout (σ={BC_DEMO_NOISE}): {len(states_demo)} / {MAX_STEPS} steps")

plot_pca_coverage(
    X6_agg, overlay=states_demo,
    overlay_label=f"SINDy+DAgger rollout (σ={BC_DEMO_NOISE})",
    overlay_color="crimson",
    title="PCA — DAgger aggregated dataset vs failing rollout",
)